# 042 数据清洗：微博数据

In [ ]:
TOPIC_WEIBO_PATH = r'..\data\features\topic_weibo.parquet'
USER_WEIBO_PATH = r'..\data\features\user_weibo.parquet'

TOPIC_COMMENT_PATH = r"..\data\cleaned\topic_comment.parquet"

In [ ]:
import pandas as pd

df_topic_weibo = pd.read_parquet(TOPIC_WEIBO_PATH)
df_user_weibo = pd.read_parquet(USER_WEIBO_PATH)

df_topic_comment = pd.read_parquet(TOPIC_COMMENT_PATH)

## `df_user_weibo` 清洗

In [ ]:
# ========== 1.1 去重 ==========
# user_weibo: 68,809 组同用户重复 weibo_id
# 策略：对于同一 weibo_id，保留第一条（同用户重复取其一）；
#        对于多用户同 weibo_id（82 组，转发关系），按 user_id 区分后保留
n_before = len(df_user_weibo)
df_user_weibo = df_user_weibo.drop_duplicates(subset=["weibo_id", "user_id"], keep="first")
print(f"\nuser_weibo 去重前：{n_before:,}  去重后: {len(df_user_weibo):>10,}")
print(f"  保留率: {len(df_user_weibo) / n_before * 100:.1f}%")

In [ ]:
# 记录表原始行数，用于全程追踪保留率
_ORIGINAL_COUNTS = {
    "topic_weibo": len(df_topic_weibo),
    "user_weibo": len(df_user_weibo),
}

def retention_rate(table: str, current_df) -> str:
    """计算相对于原始数据的保留率。"""
    orig = _ORIGINAL_COUNTS[table]
    cur = len(current_df)
    return f"{cur:,} / {orig:,} ({cur / orig * 100:.1f}% 保留)"

print("\n📦 原始数据量:")
print(f"  topic_weibo:   {len(df_topic_weibo):>10,}")
print(f"  user_weibo:    {len(df_user_weibo):>10,}")

In [ ]:
import re

def clean_text(text: str) -> str:
    """清洗微博评论文本，保留情绪信号。

    清洗步骤（有序）：
    1. 移除 HTML 标签
    2. 移除 URL
    3. 移除媒体占位内容和附加标签
       - 媒体占位内容：("xxx的微博视频/图片/音乐"等)
       - 附加标签：("@用户"等)
    4. 移除话题标签（"#热点现场#"等）
    5. 规范化空白字符
    """
    if not isinstance(text, str) or len(text) == 0:
        return text

    # 1. 移除 HTML 标签
    text = re.sub(r'<[^>]+>', '', text)

    # 2. 移除 URL
    text = re.sub(r'https?://\S+', '', text)

    # 3. 移除媒体占位内容
    # 模式：任意字符 + "的微博" + (视频|图片|音乐) 
    # 也包括其他用户名模式 + "的微博视频"等
    text = re.sub(r'\S*?的微博(?:视频|图片|音乐|文章|直播)', '', text)

    # 4. 移除@用户
    text = re.sub(r'@[^\s@]+', '', text)

    # 5. 移除话题标签
    # 通过话题标签规则，同时处理平台附加的标签
    text = re.sub(r'#([^#\r\n]+)#', '', text)

    # 6. 规范化空白（多个空白合并为一个，去首尾空白）
    text = re.sub(r'\s+', ' ', text).strip()

    return text


# 应用清洗：保留原始content，新增cleaned_content
df_user_weibo["cleaned_content"] = df_user_weibo["content"].apply(clean_text)


In [ ]:
system_patterns = [
    "此微博已被作者删除",
    "微博可见时间范围",
    "没有这条微博的查看权限",
    "账号因违反相关法律法规",
    "该微博因违反法律法规",
    "被权利方投诉侵权",
    "用户自行申请关闭", 
    "微博社区公约", 
    "暂时无法查看", 
    "账号行为异常"
]

# 平台活动关键词
activity_keywords = [
    "点开红包", "现金红包", "微博红包", "随机抽奖",
    "抽奖详情", "领取优惠券", "购买请戳", "限时特卖",
    "分享有礼", "试试你的手气", "试手气", "抽奖平台", 
    "转发评论", "转发+评论", "转发关注", "转发+关注", 
    "关注转发", "关注+转发", "转关", "转+关", "好礼", 
    "年度歌曲", "我在参与", "免费围观", "森林驿站", 
    "开放公测", "上闲鱼", "微博智搜", "微博抓马", 
    "春节AI合拍", "微博渔场", "解锁赛博年味", "年度报告", 
    "旅行青蛙中国", "微博之夜", "粉丝福利", "好运在此", 
    "运气好到爆", "嗨抢", "欧气爆棚", "抓马福", "马年接福", 
    "微博回忆", "集福袋", "惊喜福利", "独家首播", "超话大赏", 
    "爆款剧王", "星品", "微博云包场", "复制口令", "红包活动", 
    "微博直播", "派发红包", "抽奖活动", "答题挑战", "心动卡", 
    "专属权益", "赢红包", "微博年度演出", "参与抽奖"
]

# 签到打卡关键词
streak_keywords = [
    "连续签到", "粉打卡", "关注超话", "签到活动", "集卡", 
    "头像挂件", "SVIP", "微博会员", 
]
print("✅ 系统提示词 & 广告关键词 已定义")

In [ ]:
# ========== 2.2 新增 cleaned_text_length 字段 ==========
df_user_weibo["cleaned_text_length"] = df_user_weibo["cleaned_content"].str.len()

print(f"✅ user_weibo 新增 cleaned_text_length 字段")
print(f"   cleaned_text_length 统计信息：")
print(f"     均值: {df_user_weibo['cleaned_text_length'].mean():.2f}")
print(f"     中位数: {df_user_weibo['cleaned_text_length'].median():.2f}")
print(f"     最大值: {df_user_weibo['cleaned_text_length'].max()}")

# ========== 3. 文本质量分级机制 ==========
# 本阶段为 df_user_weibo 添加文本质量等级字段，用于区分微博文本分析价值
# 
# 级别定义：
# - Level 0 "系统提示" / "空内容"：无有效用户表达
#   - "系统提示"：平台生成的系统提示（删帖、权限等）
#   - "空内容"：清洗后为空的微博文本
# - Level 1 "广告、抽奖、营销等"：虽为用户文本，但主要用于商业推广、活动参与，模板化程度高
# - Level 2 "低信息量"：属于用户文本，但内容极少，信息密度极低（纯数字/符号/字母/Emoji）
# - Level 3 "普通内容"：具有基本语义内容的普通微博，可供后续情绪分析使用
# - Level 4 "高质量内容"：当前不需要识别（暂留作未来扩展）

# ========== 3.1 初始化质量等级字段 ==========
df_user_weibo["text_quality"] = 3  # 默认设为 Level 3（普通内容）
df_user_weibo["text_quality_label"] = "普通内容"

print(f"\n✅ user_weibo 初始化质量等级字段")
print(f"   Shape: {df_user_weibo.shape}")

# ========== 3.2 定义质量判定函数 ==========

# 规则集：完全匹配的正则表达式（按优先级排序）
QUALITY_RULES = [
    # Level 1: 占位符和功能互动
    (r'^\s*('
     r'[转轉][发發]?(至?微博)?|'
     r'[转轉][发發]微博\s*查看图片|'
     r'Repost|'
     r'分享(图片|新鲜事|视频)|'
     r'网页链接|'
     r'查看图片|'
     r'评论配图|'
     r'[存码马](住|下|克)?|'
     r'转一个|必须转|'
     r'签到|'
     r'收藏(了)?'
     r')\s*$', 1, '占位/功能互动'),
    (r'^#[^#]+#(\s*#[^#]+#)*$', 1, '纯话题占位'),
    
    # Level 2: 参与互动和祝福
    (r'^(我?来[了啦]{0,2}|接{1,3}|[抽中]|了解一下|收到|是的|关注|期待)$', 2, '参与互动'),
    (r'^((新年|元宵节|生日|除夕)快乐[！!]?|开工大吉|[早晚]安|早上好)$', 2, '固定祝福语'),
    (r'^(好(的|好好)?|哈{2,}|哇(哦)?|嗯)$', 2, '纯感叹'),
    (r'[\U0001F300-\U0001F9FF\U00002600-\U000027BF'
     r'\U0001FA00-\U0001FA6F\U0001FA70-\U0001FAFF\uFE00-\uFE0F\u200D]+', 2, '纯表情')

]

def classify_text_quality(text: str) -> tuple:
    """根据微博文本内容，判定其质量等级。
    
    Args:
        text (str): 微博文本内容
    
    Returns:
        tuple: (quality_level: int, quality_label: str)
    """
    
    # 先检查是否为空内容（清洗后为空字符串）
    if not isinstance(text, str) or len(text.strip()) == 0:
        return (0, "空内容")
    
    t = text.strip()
    
    # Level 0：系统提示词（平台生成，无有效用户表达）
    for pattern in system_patterns:
        if pattern in t:
            return (0, "系统提示")
    
    # Level 1：广告、抽奖、营销等（模板化内容，商业导向）
    # 平台活动模板
    for keyword in activity_keywords:
        if keyword in t:
            return (1, "平台活动模板")
    # 签到打卡模板
    for keyword in streak_keywords:
        if keyword in t:
            return (1, "签到打卡模板")
    
    # ========== 新增：完全匹配的规则（按优先级） ==========
    for pattern, level, label in QUALITY_RULES:
        if re.fullmatch(pattern, t):
            return (level, label)

    # Level 2：低信息量文本（用户文本，但内容极少）
    if _is_low_info_text(t):
        return (2, "低信息量")
    
    # Level 3：普通内容（默认，具有基本语义内容）
    return (3, "普通内容")


def _is_low_info_text(text: str) -> bool:
    """判断文本是否为低信息量（纯数字/纯符号/纯英文字母/纯Emoji/纯@用户/纯视频媒体占位）。
    
    Args:
        text (str): 已去空格的文本
    
    Returns:
        bool: True 表示低信息量，False 表示有信息量
    """
    # 纯数字
    if re.fullmatch(r'\d+', text):
        return True
    # 纯符号（不含字母、数字、汉字、Emoji）
    if re.fullmatch(r'[^\w\u4e00-\u9fff\U00010000-\U0010FFFF]+', text):
        return True
    # 纯英文字母
    if re.fullmatch(r'[a-zA-Z]+', text):
        return True
    # 纯@用户
    if re.fullmatch(r'(@[^\s@]+\s*)+', text):
        return True
    # 纯视频媒体占位
    if re.fullmatch(r'^分享视频\s*\n.+?的微博视频\s*$', text):
        return True
    return False


# ========== 3.4 应用质量分级 ==========
print("\n\n📊 应用文本质量分级...")
df_user_weibo[["text_quality", "text_quality_label"]] = df_user_weibo["content"].apply(
    lambda x: pd.Series(classify_text_quality(x))
)

# 统计各质量等级的数量
quality_counts = df_user_weibo["text_quality_label"].value_counts()
print("\n📈 文本质量等级分布：")
for quality_idx in range(5):
    quality_labels = {
        0: "空内容 / 系统提示",
        1: "广告、抽奖、营销等",
        2: "低信息量 / 参与互动 / 祝福语 / 表情",
        3: "普通内容",
        4: "高质量内容"
    }
    label = quality_labels[quality_idx]
    count = len(df_user_weibo[df_user_weibo["text_quality"] == quality_idx])
    if count > 0:
        pct = count / len(df_user_weibo) * 100
        # 对于 Level 0，显示细分信息
        if quality_idx == 0:
            empty_count = len(df_user_weibo[df_user_weibo["text_quality_label"] == "空内容"])
            system_count = len(df_user_weibo[df_user_weibo["text_quality_label"] == "系统提示"])
            print(f"  Level {quality_idx} ({label:20s}): {count:>10,} ({pct:5.2f}%)")
            print(f"       ├─ 空内容: {empty_count:>10,}")
            print(f"       └─ 系统提示: {system_count:>10,}")
        else:
            print(f"  Level {quality_idx} ({label:20s}): {count:>10,} ({pct:5.2f}%)")

# 显示新增规则的分布
print("\n📊 Level 1 标签分布（新增规则）:")
level1_labels = df_user_weibo[df_user_weibo["text_quality"] == 1]["text_quality_label"].value_counts()
for label, count in level1_labels.items():
    pct = count / len(df_user_weibo[df_user_weibo["text_quality"] == 1]) * 100
    print(f"  {label:20s}: {count:>10,} ({pct:5.2f}%)")

print("\n📊 Level 2 标签分布（新增规则）:")
level2_labels = df_user_weibo[df_user_weibo["text_quality"] == 2]["text_quality_label"].value_counts()
for label, count in level2_labels.items():
    pct = count / len(df_user_weibo[df_user_weibo["text_quality"] == 2]) * 100
    print(f"  {label:20s}: {count:>10,} ({pct:5.2f}%)")

print(f"\n✅ 文本质量分级完成: {retention_rate('user_weibo', df_user_weibo)}")


## `df_topic_weibo` 清洗

In [ ]:
df_topic_weibo = pd.read_parquet(TOPIC_WEIBO_PATH)

In [ ]:
# ========== 1.1 去重 ==========
# user_weibo: 68,809 组同用户重复 weibo_id
# 策略：对于同一 weibo_id，保留第一条（同用户重复取其一）；
#        对于多用户同 weibo_id（82 组，转发关系），按 user_id 区分后保留
n_before = len(df_topic_weibo)
df_topic_weibo = df_topic_weibo.drop_duplicates(subset=["weibo_id", "user_id"], keep="first")
print(f"\ntopic_weibo 去重前：{n_before:,}  去重后: {len(df_topic_weibo):>10,}")
print(f"  保留率: {len(df_topic_weibo) / n_before * 100:.1f}%")

In [ ]:
df_topic_weibo["cleaned_content"] = df_topic_weibo["content"].apply(clean_text)


In [ ]:
# ========== 4. 评论聚合统计并回填到 topic_weibo ==========
print("\n\n📊 开始评论聚合统计...")

# 4.1 按 weibo_id 聚合评论数据
comment_agg = df_topic_comment.groupby("weibo_id").agg(
    comment_crawled_count=("weibo_id", "size"),  # 爬取到的评论总数
    comment_hq_count=("text_quality", lambda x: (x >= 3).sum()),  # 高质量评论数
    comment_hq_user_count=("user_id", lambda x: x[df_topic_comment.loc[x.index, "text_quality"] >= 3].nunique())  # 高质量评论的去重用户数
).reset_index()

# 4.2 计算高质量评论比例
comment_agg["comment_hq_ratio"] = (comment_agg["comment_hq_count"] / comment_agg["comment_crawled_count"]).round(2)

print(f"\n✅ 评论聚合统计完成：{len(comment_agg):,} 个微博")
print(f"   字段：{list(comment_agg.columns)}")

# 4.3 将聚合结果回填到 df_topic_weibo
df_topic_weibo = df_topic_weibo.merge(
    comment_agg[["weibo_id", "comment_crawled_count", "comment_hq_count", "comment_hq_ratio", "comment_hq_user_count"]],
    on="weibo_id",
    how="left"
)

# 处理没有评论的微博（填充为 0 或相关值）
df_topic_weibo["comment_crawled_count"] = df_topic_weibo["comment_crawled_count"].fillna(0).astype(int)
df_topic_weibo["comment_hq_count"] = df_topic_weibo["comment_hq_count"].fillna(0).astype(int)
df_topic_weibo["comment_hq_ratio"] = df_topic_weibo["comment_hq_ratio"].fillna(0.0).round(2)
df_topic_weibo["comment_hq_user_count"] = df_topic_weibo["comment_hq_user_count"].fillna(0).astype(int)

print(f"\n✅ 评论统计已回填到 df_topic_weibo")

# ========== 4.4 计算话题价值等级 ==========
print("\n📊 开始计算话题价值等级...")

def calculate_topic_value(row):
    """根据评论统计信息，判定话题的价值等级。
    
    规则：
    - comment_crawled_count < 15: Level 0 "评论样本不足"
    - comment_crawled_count >= 15 且 comment_hq_count < 17: Level 1 "低价值"
    - comment_crawled_count >= 20 且 comment_hq_count >= 27 且 comment_hq_user_count >= 24: Level 3 "高价值"
    - 其余: Level 2 "一般价值"
    
    Args:
        row: DataFrame 的一行
    
    Returns:
        tuple: (topic_value: int, topic_value_label: str)
    """
    crawled = row["comment_crawled_count"]
    hq_count = row["comment_hq_count"]
    hq_user = row["comment_hq_user_count"]
    trending_type = row["trending_type"]
    
    # q10_crawled = df_topic_weibo["comment_crawled_count"].quantile(0.10)
    q1_hq_count = df_topic_weibo["comment_hq_count"].quantile(0.25)
    q3_hq_count = df_topic_weibo["comment_hq_count"].quantile(0.75)
    q1_hq_user = df_topic_weibo["comment_hq_user_count"].quantile(0.25)
    q3_hq_user = df_topic_weibo["comment_hq_user_count"].quantile(0.75)
    # q1_hq_ratio = df_topic_weibo["comment_hq_ratio"].quantile(0.25)


    # if crawled < q10_crawled:
    #     return (0, "评论样本不足")
    if hq_count < q1_hq_count or hq_user < q1_hq_user:
        return (1, "相对低价值")
    elif trending_type is not None and hq_count >= q3_hq_count and hq_user >= q3_hq_user:
        return (3, "相对高价值")
    else:
        return (2, "一般价值")

# 应用话题价值计算
df_topic_weibo[["topic_value", "topic_value_label"]] = df_topic_weibo.apply(
    lambda row: pd.Series(calculate_topic_value(row)),
    axis=1
)

# 统计话题价值分布
print(f"\n📈 话题价值等级分布：")
value_counts = df_topic_weibo["topic_value_label"].value_counts().sort_index()
for label in ["相对低价值", "一般价值", "相对高价值"]:
    count = len(df_topic_weibo[df_topic_weibo["topic_value_label"] == label])
    if count > 0:
        pct = count / len(df_topic_weibo) * 100
        print(f"  {label:15s}: {count:>10,} ({pct:5.2f}%)")

print(f"\n✅ 话题价值等级计算完成")

# 4.5 调整列顺序
topic_weibo_cols = [
    # ID & 用户
    "weibo_id", "user_id", "screen_name", "gender",
    # 话题
    "topic",
    # 文本
    "content", "cleaned_content", "text_length",
    # 时间
    "create_time", "year", "month", "day", "hour", "weekday",
    # 互动
    "like_count", "comment_count", "repost_count", "engagement",
    # 爬取评论信息
    "comment_crawled_count", "comment_hq_count", "comment_hq_ratio", "comment_hq_user_count",
    # 话题价值评估
    "topic_value", "topic_value_label",
    # 热搜词条信息
    "trending_date", "trending_type", "trending_click"
]

df_topic_weibo = df_topic_weibo[topic_weibo_cols]

print(f"\n✅ 列顺序已调整")
print(f"   新的 df_topic_weibo shape: {df_topic_weibo.shape}")
print(f"   新的 df_topic_weibo columns: {list(df_topic_weibo.columns)}")


In [ ]:
def sample_weibo_comments(level, n_weibo=50, n_comment=20) -> dict:
    """根据话题价值等级，从 df_topic_comment 中随机抽取微博评论样本。

    Args:
        level (int): 话题价值等级
        n_weibo (int): 抽样微博数量
        n_comment (int): 抽样评论数量

    Returns:
        dict
    """
    weibo_comment_dict = {}
    # 根据话题价值等级筛选微博
    level_weibo_count = len(df_topic_weibo[df_topic_weibo["topic_value"] == level])
    n_weibo = min(n_weibo, level_weibo_count)  # 确保抽样数量不超过可用微博数

    sampled_weibo_ids = df_topic_weibo[df_topic_weibo["topic_value"] == level].sample(n=n_weibo)["weibo_id"]
    for weibo_id in sampled_weibo_ids:
        comment_count = len(df_topic_comment[df_topic_comment["weibo_id"] == weibo_id])
        n_comment = min(n_comment, comment_count)  # 确保抽样数量不超过可用评论数
        sampled_comments = df_topic_comment[df_topic_comment["weibo_id"] == weibo_id][["screen_name", "content", "text_quality_label"]].sample(n=n_comment)
        weibo_comment_dict[weibo_id] = [(row["screen_name"], row["content"], row["text_quality_label"]) for _, row in sampled_comments.iterrows()]

    # 随机抽样
    return weibo_comment_dict

# sample_weibo_comments(3)

In [ ]:
import os

# 创建输出目录
output_dir = r"..\data\cleaned"
os.makedirs(output_dir, exist_ok=True)

# ========== 调整 df_user_weibo 列顺序 ==========
user_weibo_cols = [
    # ID & 用户
    "weibo_id", "user_id", "screen_name", 
    # 文本
    "content", "cleaned_content", "text_length", "cleaned_text_length",
    # 文本质量
    "text_quality", "text_quality_label",
    # 时间
    "create_time", "year", "month", "day", "hour", "weekday",
    # 互动
    "like_count", "comment_count", "repost_count", "engagement",
    # 转发关系
    "is_repost", "reposted_weibo_id", 
    # 社交原数据
    "topics", "at_users"
]

# 检查并只保留存在的列
user_weibo_cols = [col for col in user_weibo_cols if col in df_user_weibo.columns]
df_user_weibo = df_user_weibo[user_weibo_cols]

print(f"✅ df_user_weibo 列顺序已调整")
print(f"   新的 df_user_weibo shape: {df_user_weibo.shape}")
print(f"   新的 df_user_weibo columns: {list(df_user_weibo.columns)}")

# 保存
datasets = {
    "topic_weibo": df_topic_weibo,
    "user_weibo": df_user_weibo,
}

for name, df in datasets.items():
    path = os.path.join(output_dir, f"{name}.parquet")
    df.to_parquet(path, index=False)
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f"✅ {name}.parquet 已保存 ({df.shape[0]:>10,} rows × {df.shape[1]:>2} cols, {size_mb:.1f} MB)")

print(f"\n📂 输出目录: {os.path.abspath(output_dir)}")

# 显示 user_weibo 的最终字段列表
print(f"\n📋 user_weibo 最终字段列表：")
print(f"  Columns: {list(df_user_weibo.columns)}")


## 📊 微博文本质量等级体系总结

### 等级定义与文本种类

#### **Level 0: 系统提示 / 空内容** ❌
**特征**：无有效用户表达
- **系统提示**（占比中的一部分）
  - 微博被作者删除："此微博已被作者删除"
  - 权限限制："微博可见时间范围"、"没有这条微博的查看权限"
  - 账号问题："账号因违反相关法律法规"、"账号行为异常"
  - 内容违规："该微博因违反法律法规"、"被权利方投诉侵权"
  - 其他："用户自行申请关闭"、"微博社区公约"、"暂时无法查看"
  
- **空内容**（占比中的一部分）
  - 清洗后为空字符串的微博文本
  - 无法分析的无效数据

**应用场景**：应被过滤掉，不用于后续情绪分析

---

#### **Level 1: 广告、抽奖、营销等** 🎁
**特征**：虽为用户文本，但主要用于商业推广、活动参与，模板化程度高

**三大子分类**：

1. **平台活动模板**
   - 抽奖红包类：红包、抽奖、中奖、现金、优惠券、免费、试手气等
   - 互动参与类：转发评论、转发关注、转关、分享等指令
   - 营销推广类：限时特卖、购买请戳、年度报告等

2. **签到打卡模板**
   - 连续签到、打卡、超话关注、集卡
   - 虚拟商品：头像挂件、SVIP、微博会员等

3. **占位符和功能互动**（通过完全匹配规则识别）
   - 分享占位：转发、转发微博、Repost、分享图片/视频
   - 媒体占位：网页链接、查看图片、评论配图
   - 收藏/存储指令：存下、存克、码住、收藏等

4. **纯话题占位**
   - 只包含话题标签，无其他内容（如 `#热点现场# #新闻#`）

**应用场景**：用户互动积极但内容价值低，可选择过滤或降权处理

---

#### **Level 2: 低信息量** 💬
**特征**：属于用户文本，但内容极少，信息密度极低

**五大子分类**：

1. **参与互动**
   - 单词应答：我来了、接、接接接、了解一下、收到、是的、关注、期待
   - 被动应答，无核心观点或情感表达

2. **固定祝福语**
   - 节日祝福：新年快乐、元宵节快乐、生日快乐、除夕快乐等
   - 日常问候：早安、晚安、早上好、开工大吉等
   - 高度模板化，情绪表达重复

3. **纯感叹**
   - 无语言文本内容：好、好的、好好好、哈哈哈、哇、嗯等
   - 仅传达情感反应，无具体观点

4. **纯表情** 😊
   - 仅由 Emoji 组成的文本
   - 无文字信息，难以进行语义分析

5. **纯特殊字符**
   - 仅包含数字、符号、英文字母等
   - 纯数字：123456
   - 纯符号：！！！？？？
   - 纯英文字母：ABC

**应用场景**：可用于互动度分析，但不适合情绪分析；可考虑单独分类处理

---

#### **Level 3: 普通内容** 📝
**特征**：具有基本语义内容的普通微博，具有一定信息密度

**包含范围**：
- 日常观点与评论
- 新闻转载与讨论
- 生活分享与记录
- 普通问答与对话
- 信息发布与通知

**特征**：
- 经过清洗后保留完整意义的文本
- 不属于 Level 0-2 的所有用户生成内容
- 具备基本语义可分析性

**应用场景**：适合进行情绪分析、观点挖掘、信息扩散分析等

---

#### **Level 4: 高质量内容** ⭐
**特征**：当前不需要识别（暂留作未来扩展）

**预期特征**（未来可能包含）：
- 深度观点与分析
- 高质量讨论与对话
- 创意内容与原创观点
- 信息价值高的内容

---

### 🔍 识别顺序（优先级）

代码中识别的顺序如下：

1. **空内容检查** → 长度为 0 的文本
2. **系统提示词匹配** → 包含特定的平台系统提示
3. **平台活动关键词** → 包含活动、红包、抽奖等关键词
4. **签到打卡关键词** → 包含签到、打卡等关键词
5. **完全匹配规则**（按优先级）
   - 占位/功能互动
   - 纯话题占位
   - 参与互动
   - 固定祝福语
   - 纯感叹
   - 纯表情
6. **低信息量检查** → 纯数字、纯符号、纯英文、纯@用户等
7. **默认分类** → 普通内容（Level 3）

---

### 📈 质量等级分布特点

- **Level 0** 占比较小：代表无效数据，应完全过滤
- **Level 1** 占比中等：代表商业内容，需要选择性处理
- **Level 2** 占比可能较大：代表互动参与，价值有限
- **Level 3** 占比最大：代表主要有效内容，是情绪分析的主要来源

---

### 🎯 应用建议

| 质量等级 | 情绪分析 | 信息传播 | 用户互动 | 其他分析 |
|---------|--------|--------|--------|--------|
| **Level 0** | ❌ 不用 | ❌ 不用 | ❌ 不用 | ❌ 过滤 |
| **Level 1** | ⚠️ 可选 | ⚠️ 可选 | ✅ 记录 | ⚠️ 分离 |
| **Level 2** | ⚠️ 可选 | ⚠️ 可选 | ✅ 用 | ✅ 互动度 |
| **Level 3** | ✅ 主用 | ✅ 主用 | ✅ 用 | ✅ 主用 |
| **Level 4** | ✅ 优先 | ✅ 优先 | ✅ 优先 | ✅ 优先 |